In [6]:
# CELL 1 — Imports & config
import pandas as pd
import os

BASE_PATH   = r"C:\Users\Lenovo\Downloads\nyc_taxi_project"
BRONZE_PATH = os.path.join(BASE_PATH, "bronze", "yellow_trips_bronze.parquet")
SILVER_DIR  = os.path.join(BASE_PATH, "silver")

os.makedirs(SILVER_DIR, exist_ok=True)

In [7]:
# CELL 2 — Load Bronze
df = pd.read_parquet(BRONZE_PATH)
print(f"Bronze records loaded: {len(df):,}")
display(df.head(3))

Bronze records loaded: 3,066,766


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0


In [8]:
# CELL 3 — Check nulls
key_cols = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "trip_distance", "fare_amount", "passenger_count"
]
print("Null counts in key columns:")
print(df[key_cols].isnull().sum())

Null counts in key columns:
tpep_pickup_datetime         0
tpep_dropoff_datetime        0
trip_distance                0
fare_amount                  0
passenger_count          71743
dtype: int64


In [9]:
# CELL 4 — Select & rename columns
df = df[[
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance",
    "PULocationID", "DOLocationID",
    "fare_amount", "tip_amount",
    "total_amount", "payment_type", "RatecodeID"
]].rename(columns={
    "tpep_pickup_datetime" : "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "trip_distance"        : "trip_distance_miles",
    "PULocationID"         : "pickup_location_id",
    "DOLocationID"         : "dropoff_location_id",
    "RatecodeID"           : "rate_code"
})

# Fix data types
df["pickup_datetime"]  = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])
df["passenger_count"]  = df["passenger_count"].astype("Int64")
df["payment_type"]     = df["payment_type"].astype("Int64")
df["rate_code"]        = df["rate_code"].astype("Int64")

print("Columns renamed and types fixed!")
print(df.dtypes)

Columns renamed and types fixed!
pickup_datetime        datetime64[us]
dropoff_datetime       datetime64[us]
passenger_count                 Int64
trip_distance_miles           float64
pickup_location_id              int64
dropoff_location_id             int64
fare_amount                   float64
tip_amount                    float64
total_amount                  float64
payment_type                    Int64
rate_code                       Int64
dtype: object


In [10]:
# CELL 5 — Filter invalid records
before = len(df)

df = df[
    df["pickup_datetime"].notna() &
    df["dropoff_datetime"].notna() &
    (df["dropoff_datetime"] > df["pickup_datetime"]) &
    (df["trip_distance_miles"] > 0) &
    (df["fare_amount"] > 0) &
    (df["total_amount"] > 0) &
    (df["passenger_count"] >= 1) &
    (df["trip_distance_miles"] <= 200) &
    (df["fare_amount"] <= 500)
]

print(f"Before cleaning : {before:,}")
print(f"After cleaning  : {len(df):,}")
print(f"Removed         : {before - len(df):,} bad records")

Before cleaning : 3,066,766
After cleaning  : 2,884,130
Removed         : 182,636 bad records


In [11]:
# CELL 6 — Add derived columns
df["trip_duration_minutes"] = (
    (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds() / 60
).round(2)

df["avg_speed_mph"] = (
    df["trip_distance_miles"] / (df["trip_duration_minutes"] / 60)
).round(2)

df["tip_percentage"]     = ((df["tip_amount"] / df["fare_amount"]) * 100).round(2)
df["pickup_hour"]        = df["pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["pickup_datetime"].dt.dayofweek  # 0=Monday, 6=Sunday
df["pickup_date"]        = df["pickup_datetime"].dt.date

df["payment_type_label"] = df["payment_type"].map({
    1: "Credit Card", 2: "Cash",
    3: "No Charge",   4: "Dispute"
}).fillna("Unknown")

# Remove impossible values
df = df[
    (df["avg_speed_mph"] > 0) &
    (df["avg_speed_mph"] <= 150) &
    (df["trip_duration_minutes"] > 0)
]

print(f"Final Silver records: {len(df):,}")
display(df[["pickup_datetime", "trip_duration_minutes", "avg_speed_mph", "tip_percentage"]].head())

Final Silver records: 2,883,081


,pickup_datetime,trip_duration_minutes,avg_speed_mph,tip_percentage
0,2023-01-01 00:32:10,8.43,6.90,0.00
1,2023-01-01 00:55:08,6.32,10.44,50.63
2,2023-01-01 00:25:04,12.75,11.81,100.67
4,2023-01-01 00:10:29,10.83,7.92,28.77
5,2023-01-01 00:50:34,12.30,8.98,78.12


In [12]:
# CELL 7 — Save Silver
SILVER_PATH = os.path.join(SILVER_DIR, "yellow_trips_silver.parquet")
df.to_parquet(SILVER_PATH, index=False)

size_mb = os.path.getsize(SILVER_PATH) / 1024 / 1024
print(f"Silver saved!")
print(f"Path : {SILVER_PATH}")
print(f"Size : {size_mb:.1f} MB")

Silver saved!
Path : C:\Users\Lenovo\Downloads\nyc_taxi_project\silver\yellow_trips_silver.parquet
Size : 67.6 MB
